In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
from copy import deepcopy

from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from bait.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from bait.core import bait_prompts

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
out_dir = f'{data_dir}/check_zero_shot'

in_file_path = f'{data_dir}/raw_datasets/Knowledge-Editing/mcf/multi_counterfact_identical1_all_19366.json'
datas = json_utils.load_json(in_file_path)

In [ ]:
dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

In [ ]:
def check_zero_shot(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast, datas, batch_size, out_prefix):
    zero_shot_facts, zero_shot_counters, zero_shot_others = [], [], []

    cnt = 0
    for datas_batch in container_utils.chunks(datas, batch_size):
        prompts_batch = []

        for data in datas_batch:
            prompt = data['requested_rewrite']['prompt']
            subject = data['requested_rewrite']['subject']
            question = prompt.format(subject)
            prompts_batch.append(bait_prompts.get_generate_prompt(question))
        
        generated_texts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts_batch, max_seq_length, max_new_tokens
        )

        for generated_text, data in zip(generated_texts, datas_batch):
            answer_fact = data['requested_rewrite']['target_true']['str']
            answer_counter = data['requested_rewrite']['target_new']['str']
            data['requested_rewrite']['generated_text'] = generated_text

            if model_utils.is_correct(generated_text, answer_fact)[1]:
                zero_shot_facts.append(data)
            elif model_utils.is_correct(generated_text, answer_counter)[1]:
                zero_shot_counters.append(data)
            else:
                zero_shot_others.append(data)
        
        cnt += 1
        if (cnt % 20) == 0:
            print(f'check_zero_shot() {cnt*batch_size} complet')
    print(f'check_zero_shot() {len(datas)} complet')
    
    print(f'\ncheck_zero_shot() zero_shot_fact size : {len(zero_shot_facts)}')
    print(f'check_zero_shot() zero_shot_counter size : {len(zero_shot_counters)}')
    print(f'check_zero_shot() zero_shot_other size : {len(zero_shot_others)}')

    json_utils.write_json(zero_shot_facts, f'{out_prefix}_fact.json')
    json_utils.write_json(zero_shot_counters, f'{out_prefix}_counter.json')
    json_utils.write_json(zero_shot_others, f'{out_prefix}_other.json')

In [ ]:
model_names = ['Llama-3.2-3B', 'Llama-3.1-8B']

for model_name in model_names:
    if model_name.startswith(f'Llama'):
        model_name_or_path = f'meta-llama/{model_name}-Instruct'
    else:
        model_name_or_path = model_name
    
    model = model_utils.get_model(model_name_or_path, dtype, device=device, is_eval=True)

    # 평가/추론 시에는 반드시 'left' 패딩
    tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

    out_prefix = f'{out_dir}/{model_name}/bait_{model_name}_checked_zero_shot'
    check_zero_shot(model, tokenizer, datas, 100, out_prefix)

    del model
    del tokenizer
    common_utils.clear_gpu_memory()